# Lektion 13 - Agenten-Speicher mit Cognee Knowledge Graphs


## Einrichtung

Dieses Notebook zeigt, wie man einen intelligenten **Coding-Assistenten** mit persistentem Gedächtnis unter Verwendung von [**Cognee**](https://www.cognee.ai/) Wissensgraphen und dem **Microsoft Agent Framework** (MAF) erstellt.

Cognee wandelt unstrukturierte Texte in einen strukturierten, abfragbaren Wissensgraphen um, der von Vektor-Einbettungen unterstützt wird — wodurch Ihr Agent ein reichhaltiges, beziehungsbewusstes Langzeitgedächtnis erhält.

### Was Sie lernen werden
1. **Wissensgraphen erstellen**: Entwicklerprofile und Best Practices in strukturiertes, abfragbares Wissen umwandeln.
2. **Integration von Cognee mit MAF**: `@tool`-Funktionen verwenden, damit ein MAF-Agent den Cognee-Wissensgraphen abfragen kann.
3. **Sitzungsbewusste Gespräche**: Kontext über mehrere Fragen in derselben Sitzung aufrechterhalten.
4. **Langzeitgedächtnis**: Wichtige Wissensinhalte über Sitzungen hinweg persistieren und in neuen Gesprächen abrufen.

### Voraussetzungen
- Python 3.9+
- Redis lokal laufend (`docker run -d -p 6379:6379 redis`) für Sitzungsverwaltung
- Einen LLM-API-Schlüssel (z.B. OpenAI) — in `.env` `LLM_API_KEY` setzen
- `CACHING=true` in `.env` (erforderlich für Cognee-Sitzungen)
- Ein Azure AI Foundry-Projekt mit einem bereitgestellten Chatmodell
- `AZURE_AI_PROJECT_ENDPOINT` und `AZURE_AI_MODEL_DEPLOYMENT_NAME` in `.env`
- Azure CLI authentifiziert (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Arten von Agentenspeicher

Dieses Notebook untersucht dieselben drei Speicherarten wie im Haupt-Notebook zu Lektion 13, verwendet jedoch Cognee als Langzeitspeicher-Backend:

| Speichertyp | Mechanismus | Lebensdauer |
|---|---|---|
| **Arbeits** | `agent.create_session()` (MAF) | Einzelne Konversation |
| **Kurzzeit** | Cognee Session-Cache (Redis) | Einzelne Sitzung |
| **Langzeit** | Cognee Wissensgraph + Vektoren | Permanent |

### Cognees Speicherarchitektur
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Cognee Speicher vorbereiten


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Teil 1 — Aufbau der Wissensdatenbank

Wir erfassen drei Arten von Daten, um eine umfassende Wissensdatenbank für unseren Coding-Assistenten zu erstellen:

1. **Entwicklerprofil** — persönliche Expertise und technischer Hintergrund  
2. **Python Best Practices** — das Zen von Python mit praktischen Richtlinien  
3. **Historische Unterhaltungen** — frühere Q&A-Sitzungen zwischen Entwicklern und KI-Assistenten


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## Visualisieren des Wissensgraphen

Cognee kann eine interaktive HTML-Visualisierung der extrahierten Entitäten und Beziehungen darstellen.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Speicher mit Memify anreichern

`memify()` analysiert den Wissensgraphen und erzeugt intelligente Regeln — dabei werden Muster, bewährte Verfahren und Beziehungen zwischen Konzepten identifiziert.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Teil 2 — MAF-Agent mit Cognee-Tools

Jetzt erstellen wir einen MAF-Agenten, der die Wissensgraphen von Cognee über `@tool`-Funktionen abfragen kann. Dadurch kann der Agent die volle Leistung der graphbasierten semantischen Suche nutzen und dabei den Konversationskontext durch Sitzungen aufrechterhalten.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## Arbeitsgedächtnis mit Sitzungen

Die `AgentSession` (erstellt über `agent.create_session()`) bietet ein Arbeitsgedächtnis innerhalb einer Sitzung. Der Agent kann sich auf frühere Nachrichten beziehen und gleichzeitig das langfristige Wissensnetzwerk von Cognee abfragen.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## Neue Sitzung — Langzeitgedächtnis bleibt erhalten

Das Starten einer neuen Sitzung löscht das Arbeitsgedächtnis, aber der Cognee-Wissensgraph ist weiterhin verfügbar. Der Agent kann dasselbe Langzeitwissen in einem völlig neuen Gespräch abrufen.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Zusammenfassung

In diesem Notebook haben Sie einen Code-Assistenten erstellt, der **MAFs Arbeitsgedächtnis** (`agent.create_session()`) mit **Cognees langfristigem Wissensgraphen** kombiniert.

### Was Sie gelernt haben
1. **Wissensgraph-Konstruktion**: Cognee verarbeitet unstrukturierte Texte und erstellt einen Graphen + Vektor-Speicher.
2. **Graph-Anreicherung mit memify**: Abgeleitete Fakten und reichhaltigere Beziehungen auf Ihrem bestehenden Graphen.
3. **MAF + Cognee-Integration**: `@tool`-Funktionen ermöglichen es einem MAF-Agenten, den Cognee-Graphen auf natürliche Weise abzufragen.
4. **Arbeitsgedächtnis + Langzeitgedächtnis**: `AgentSession` (über `agent.create_session()`) stellt Sitzungs-Kontext bereit, während Cognee persistentes Wissen bietet.
5. **Gefilterte Suche mit NodeSets**: Zielgerichtete Teilmengen des Wissensgraphen abfragen (z. B. nur Prinzipien).

### Wesentliche Erkenntnisse
- **Cognee** verwandelt Rohtexte in strukturiertes, beziehungsbewusstes Gedächtnis — mächtiger als ein simpler Vektor-Speicher.
- **`@tool`-Funktionen** verbinden MAF-Agenten sauber mit externen Wissenssystemen.
- **`AgentSession`** (über `agent.create_session()`) hält kontextbezogene Informationen pro Gespräch getrennt vom langfristigen Wissen.
- Derselbe Wissensgraph dient mehreren Sitzungen und Agenten.

### Anwendungsbeispiele in der Praxis
- **Entwickler-Copiloten**: Code-Reviews, Vorfallanalysen, Architekturassistenten
- **Kundenorientierte Copiloten**: Support-Agenten für Produktdokumentationen, FAQs und CRM-Notizen
- **Interne Experten-Copiloten**: Assistenten für Richtlinien, Recht oder Sicherheit, die über Leitlinien argumentieren
- **Einheitliche Datenschichten**: Kombination strukturierter und unstrukturierter Daten in einem abfragbaren Graphen

### Nächste Schritte
- Mit temporalem Bewusstsein in Cognee experimentieren
- Definition einer OWL-Ontologie für domänenspezifische Graphqualität
- Feedbackschleifen der Nutzer hinzufügen, um die Abfragequalität im Laufe der Zeit zu verbessern
- Skalierung auf Multi-Agenten-Systeme, die dieselbe Cognee-Gedächtnisschicht teilen


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Haftungsausschluss**:  
Dieses Dokument wurde mit dem KI-Übersetzungsdienst [Co-op Translator](https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir auf Genauigkeit achten, kann es bei automatischen Übersetzungen zu Fehlern oder Ungenauigkeiten kommen. Das Originaldokument in seiner Ursprungssprache ist als maßgebliche Quelle zu betrachten. Für wichtige Informationen wird eine professionelle menschliche Übersetzung empfohlen. Wir übernehmen keine Haftung für Missverständnisse oder Fehlinterpretationen, die aus der Nutzung dieser Übersetzung entstehen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
